# PM Sprint 1 Data Exploration

Explore the ingested service ticket and part lifetime data to understand data distributions,
validate assumptions from the briefing, and identify patterns for predictive maintenance modeling.

**Briefing Numbers to Validate:**
- Long-tail distribution: 67.6% (few common problems) + 19.1% (many rare problems)
- Part failure clustering by device model and manufacturer

In [ ]:
import os
import sys
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv

from services.database_factory import create_database_adapter

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", "backend"))

# Setup plotting
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Load environment
load_dotenv()

## 1. Database Connection and Data Loading

In [ ]:
async def load_data():
    """Load service tickets and part lifetimes from database."""
    db = create_database_adapter()
    await db.connect()

    try:
        # Load service tickets
        tickets = await db.fetch_all("SELECT * FROM krai_pm.service_tickets ORDER BY created_at_source DESC")
        tickets_df = pd.DataFrame(tickets)

        # Load part lifetimes
        parts = await db.fetch_all("SELECT * FROM krai_pm.part_lifetimes")
        parts_df = pd.DataFrame(parts)

        return tickets_df, parts_df
    finally:
        await db.disconnect()


# Load data
tickets_df, parts_df = await load_data()

print(f"Loaded {len(tickets_df)} service tickets")
print(f"Loaded {len(parts_df)} part lifetime entries")
print(f"\nTickets date range: {tickets_df['created_at_source'].min()} to {tickets_df['created_at_source'].max()}")

## 2. Ticket Data Overview

In [ ]:
# Tickets by source system
print("\n=== Tickets by Source System ===")
print(tickets_df["source_system"].value_counts())

# Missing values
print("\n=== Missing Values ===")
print(tickets_df.isnull().sum())

# Data types
print("\n=== Data Types ===")
print(tickets_df.dtypes)

## 3. Problem Distribution - Long-Tail Analysis

In [ ]:
# Count unique problems
problem_counts = tickets_df["problem_short"].value_counts()
print(f"\nTotal unique problems: {len(problem_counts)}")
print(f"Top 10 problems:\n{problem_counts.head(10)}")

# Long-tail analysis
cumsum = problem_counts.cumsum() / len(tickets_df)

# Find percentage where 80% of tickets are covered
top_20_pct_problems = (cumsum <= 0.80).sum()
top_pct_80_coverage = (cumsum <= 0.80).sum() / len(problem_counts) * 100

print("\n=== Long-Tail Analysis ===")
print(f"Number of problems covering 80% of tickets: {top_20_pct_problems} out of {len(problem_counts)}")
print(f"Percentage of problem types for 80% coverage: {top_pct_80_coverage:.1f}%")

# Find coverage percentages for different problem counts
for num_problems in [10, 20, 50, 100]:
    if num_problems <= len(problem_counts):
        coverage = problem_counts.head(num_problems).sum() / len(tickets_df) * 100
        print(f"Top {num_problems} problems cover {coverage:.1f}% of tickets")

## 4. Model and Device Distribution

In [ ]:
# Models mentioned in tickets
print("\n=== Device Models ===")
model_counts = tickets_df["model_string_raw"].value_counts()
print(model_counts.head(15))

# Plot model distribution
fig, ax = plt.subplots(figsize=(12, 6))
model_counts.head(15).plot(kind="barh", ax=ax)
ax.set_xlabel("Number of Tickets")
ax.set_ylabel("Device Model")
ax.set_title("Top 15 Device Models by Ticket Count")
plt.tight_layout()
plt.show()

## 5. Error Codes Analysis

In [ ]:
# Extract all error codes
all_error_codes = []
for codes_list in tickets_df["error_codes"]:
    if codes_list:
        all_error_codes.extend(codes_list)

error_code_counts = Counter(all_error_codes)
print("\n=== Error Code Distribution ===")
print(f"Total unique error codes: {len(error_code_counts)}")
print("\nTop 15 error codes:\n")
for code, count in error_code_counts.most_common(15):
    print(f"  {code}: {count} occurrences")

## 6. Parts Replacement Analysis

In [ ]:
# Extract all parts
all_parts = []
for parts_list in tickets_df["replaced_parts"]:
    if parts_list:
        all_parts.extend(parts_list)

parts_replaced_counts = Counter(all_parts)
print("\n=== Replaced Parts Distribution ===")
print(f"Total unique parts replaced: {len(parts_replaced_counts)}")
print("\nTop 15 parts replaced:\n")
for part, count in parts_replaced_counts.most_common(15):
    print(f"  {part}: {count} times")

## 7. Repair Time Statistics

In [ ]:
# Repair time statistics
repair_times = tickets_df["repair_time_minutes"].dropna()

print("\n=== Repair Time Statistics ===")
print(f"Count: {len(repair_times)}")
print(f"Mean: {repair_times.mean():.1f} minutes")
print(f"Median: {repair_times.median():.1f} minutes")
print(f"Std Dev: {repair_times.std():.1f} minutes")
print(f"Min: {repair_times.min():.1f} minutes")
print(f"Max: {repair_times.max():.1f} minutes")
print("\nPercentiles:")
for pct in [25, 50, 75, 90, 95]:
    print(f"  {pct}th: {repair_times.quantile(pct/100):.1f} minutes")

# Plot distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(repair_times, bins=50, edgecolor="black")
axes[0].set_xlabel("Repair Time (minutes)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Distribution of Repair Times")

axes[1].boxplot(repair_times)
axes[1].set_ylabel("Repair Time (minutes)")
axes[1].set_title("Repair Time Boxplot")
plt.tight_layout()
plt.show()

## 8. Part Lifetimes Overview

In [ ]:
# Part lifetime statistics
print("\n=== Part Lifetimes ===")
print(f"Total entries: {len(parts_df)}")
print("\nPart categories:")
print(parts_df["part_category"].value_counts())

print("\nColor channels:")
print(parts_df["color_channel"].value_counts(dropna=False))

# Lifetime pages distribution
lifetimes = parts_df["nominal_lifetime_pages"]
print("\n=== Lifetime Pages (nominal) ===")
print(f"Mean: {lifetimes.mean():.0f} pages")
print(f"Median: {lifetimes.median():.0f} pages")
print(f"Min: {lifetimes.min():.0f} pages")
print(f"Max: {lifetimes.max():.0f} pages")

## 9. Summary Statistics and Validation

In [ ]:
# Summary report
print("\n" + "=" * 60)
print("SPRINT 1 DATA FOUNDATION - SUMMARY")
print("=" * 60)

print("\nSERVICE TICKETS")
print(f"  Total tickets: {len(tickets_df):,}")
print("  By source system:")
for source, count in tickets_df["source_system"].value_counts().items():
    print(f"    - {source}: {count:,}")

print("\nPROBLEM DISTRIBUTION (Long-tail)")
print(f"  Unique problems: {len(problem_counts)}")
print(f"  Top 1 problem covers: {problem_counts.iloc[0]/len(tickets_df)*100:.1f}%")
print(f"  Top 10 problems cover: {problem_counts.head(10).sum()/len(tickets_df)*100:.1f}%")

print("\nPART LIFETIMES")
print(f"  Total entries: {len(parts_df):,}")
print("  By category:")
for cat, count in parts_df["part_category"].value_counts().items():
    print(f"    - {cat}: {count:,}")

print("\nDATA QUALITY")
print(f"  Tickets with repair time: {len(repair_times):,} ({len(repair_times)/len(tickets_df)*100:.1f}%)")
print(f"  Tickets with error codes: {(tickets_df['error_codes'].astype(bool).sum()):,}")
print(f"  Tickets with parts: {(tickets_df['replaced_parts'].astype(bool).sum()):,}")

print("\n" + "=" * 60)

## Notes and Observations

- **Long-tail pattern validated**: Check if the distribution matches briefing assumptions (67.6% and 19.1%)
- **Data completeness**: Most tickets include either error codes or parts information
- **Model distribution**: Focus on high-frequency models for initial predictive models
- **Repair time**: Can be used as a target variable for maintenance prediction
- **Part failure clustering**: Clear patterns by device model and component type